In [2]:
#!/usr/bin/env python3
"""
sim_and_train.py

1) Simulate hybrid real+synthetic multi-frame FFT data (RAM-optimized streaming).
2) Train a compact Keras CNN on the generated 1-hour dataset.

Outputs (in ./data):
 - FFT_multiframe.dat     : memmap float32 array (N, T, F)
 - FFT_multiframe_meta.csv: metadata per-sample (timestamp, ap_id, band, channel, label, center_mhz, bw_mhz, duty)
 - fft_dataset.csv        : legacy 64-bin CSV (streamed)
 - sensing_events.csv     : events (streamed)
 - Aplog_all_channels.csv : AP telemetry (streamed)
 - model/                 : trained Keras model, checkpoints, training logs
"""

import os, math, random, argparse, logging, sys
from pathlib import Path
from dataclasses import dataclass
from datetime import datetime, timedelta
import numpy as np
import pandas as pd
import tensorflow as tf

# -----------------------
# Basic logging
# -----------------------
logging.basicConfig(level=logging.INFO, format="%(asctime)s [%(levelname)s] %(message)s")
log = logging.getLogger("SimTrain")

# -----------------------
# Config (tweak here)
# -----------------------
SEED = 42
random.seed(SEED); np.random.seed(SEED)
OUT = Path("data"); OUT.mkdir(exist_ok=True)
MODEL_DIR = Path("model"); MODEL_DIR.mkdir(exist_ok=True)

# Simulation params
HOURS = 6.0                  # default 1 hour as requested
SCAN_PERIOD = 10.0           # seconds per sensing tick
SENSE_TIME = 0.12
MODE = "hybrid"              # synthetic | real | hybrid
FRAC_REAL = 0.7              # for hybrid

# FFT / frames
FFT_BINS = 512               # 256 or 512 recommended (you can change)
FRAMES_PER_SAMPLE = 24       # 20-30 recommended
LEGACY_BINS = 64

# AP / environment
CH24 = list(range(1,15))
DFS_CH = [52,56,60,64,100,104,108,112,116,120,124,128,132,136,140,144]
CH5  = [36,40,44,48] + DFS_CH + [149,153,157,161]
WIDTHS = [20,40,80]
WIDTH_SCALE = {20:1.0,40:1.7,80:3.0}
PHY_MAX_24, PHY_MAX_5 = 72.2, 433.0
CLIENTS_PER_AP = 10          # reduced for speed; adjust as desired
CELL_R = 12.0

# "Real" FFT library path (optional) - set to your mounted drive if available
REAL_FFT_DIR = Path("/content/drive/MyDrive/RRM+ Arista/911_data/data/cleaned_datasets")

# Training params
BATCH_SIZE = 64
EPOCHS = 30
LR = 1e-3
IMG_HEIGHT = FRAMES_PER_SAMPLE
IMG_WIDTH = FFT_BINS

# -----------------------
# Dataclasses / helpers
# -----------------------
@dataclass
class AP:
    id: str; band: str; ch: int; width: int; tx: float; x: float; y: float; phy: float; gain: float = 2.0
@dataclass
class Client:
    id: str; apid: str; x: float; y: float; d: float

def ch2mhz(band, ch): return 2407+5*ch if band=="2.4" else 5000+5*ch
def width_scale(w): return WIDTH_SCALE.get(w,1.0)
def noise_floor_dbm(band,bw):
    base = -174 + 10*math.log10(bw*1e6)
    nf = 6.0 if band=="2.4" else 6.5
    return base + nf + np.random.normal(0,1.2)

# -----------------------
# Load real FFT library (optional)
# -----------------------
def load_real_fft_library(real_dir: Path):
    lib = {}
    if not real_dir.exists():
        log.warning("REAL_FFT_DIR not found (%s). Real-mode will be unavailable.", real_dir)
        return lib
    candidates = ["fft_dataset_24g_stage2.csv","fft_dataset_5g_stage2.csv","fft_dataset.csv"]
    dfs=[]
    for c in candidates:
        p = real_dir / c
        if p.exists():
            try:
                d = pd.read_csv(p)
                d.columns = [cc.strip() for cc in d.columns]
                dfs.append(d)
                log.info("Loaded %s (%d rows)", p.name, len(d))
            except Exception as e:
                log.warning("Failed to load %s: %s", p, e)
    if not dfs:
        return lib
    df = pd.concat(dfs, ignore_index=True)
    if "label" not in df.columns:
        log.warning("Real FFT files missing 'label' column; ignoring real library.")
        return lib
    for lab in df['label'].dropna().unique():
        lib[lab] = df[df['label']==lab].reset_index(drop=True)
    log.info("Real FFT library: %s", {k: len(v) for k,v in lib.items()})
    return lib

REAL_LIB = load_real_fft_library(REAL_FFT_DIR)

# -----------------------
# Synthetic generator (compact)
# -----------------------
def generate_highres_fft_synthetic(label, center_freq, bandwidth_mhz, noise_floor):
    freqs = np.linspace(center_freq-40, center_freq+40, FFT_BINS)
    frames = np.zeros((FRAMES_PER_SAMPLE, FFT_BINS), dtype=np.float32)
    for i in range(FRAMES_PER_SAMPLE):
        base_noise = np.random.normal(noise_floor-5, 1.5, FFT_BINS)
        if label=="BLE":
            tone_center = center_freq + np.random.uniform(-1,1)
            signal = np.exp(-((freqs-tone_center)**2)/(2*(2**2))) * 40
            frames[i] = base_noise + signal
        elif label=="ZigBee":
            tone_center = center_freq + np.random.uniform(-1,1)
            signal = np.exp(-((freqs-tone_center)**2)/(2*(3.5**2))) * 35 + np.random.normal(0,2,FFT_BINS)
            frames[i] = base_noise + signal
        elif label=="Microwave":
            bw = bandwidth_mhz + np.random.uniform(5,20)
            signal = np.exp(-((freqs-center_freq)**2)/(2*(bw**2))) * 22
            frames[i] = base_noise + signal
        elif label=="FHSS":
            hop_center = center_freq + np.random.randint(-20,20)
            signal = np.exp(-((freqs-hop_center)**2)/(2*(3**2))) * 40
            frames[i] = base_noise + signal
        elif label=="Radar":
            if np.random.rand() < 0.3:
                bw = np.random.uniform(10,30)
                pulse = np.exp(-((freqs-center_freq)**2)/(2*(bw**2))) * 60
                frames[i] = base_noise + pulse
            else:
                frames[i] = base_noise
        else:
            frames[i] = base_noise
    return frames.astype(np.float32)

# -----------------------
# Real-based augmentation generator
# -----------------------
def generate_realistic_fft_from_real_data(label, center_freq, bandwidth_mhz, noise_floor):
    if not REAL_LIB or label not in REAL_LIB:
        return generate_highres_fft_synthetic(label, center_freq, bandwidth_mhz, noise_floor)
    df = REAL_LIB[label]
    row = df.sample(1).iloc[0]
    real_bins = []
    for i in range(64):
        col = f"fft_bin_{i}"
        if col in row:
            real_bins.append(row[col])
        else:
            real_bins.append(noise_floor - 3 + np.random.normal(0,1))
    real_bins = np.array(real_bins, dtype=float)
    x_old = np.arange(64)
    x_new = np.linspace(0,63,FFT_BINS)
    real_upsampled = np.interp(x_new, x_old, real_bins).astype(np.float32)
    freqs = np.linspace(center_freq-40, center_freq+40, FFT_BINS)
    frames = np.zeros((FRAMES_PER_SAMPLE, FFT_BINS), dtype=np.float32)
    for f in range(FRAMES_PER_SAMPLE):
        base = np.random.normal(noise_floor-5, 1.2, FFT_BINS)
        shift_hz = np.random.uniform(-3,3)
        # approximate shift in bins
        total_span = freqs[-1]-freqs[0] if freqs[-1]!=freqs[0] else 1.0
        shift_bins = int((shift_hz/total_span) * FFT_BINS * 8)  # scaled
        shifted = np.roll(real_upsampled, shift_bins)
        scale = np.random.uniform(0.8,1.4)
        signal = shifted * scale
        bw = max(1.0, bandwidth_mhz + np.random.uniform(-2,2))
        envelope = np.exp(-((freqs - (center_freq + shift_hz))**2) / (2 * (bw**2)))
        shaped = signal * envelope * 1.5
        frames[f] = base + shaped + np.random.normal(0,0.5,FFT_BINS)
    return frames.astype(np.float32)

# -----------------------
# AP / client helpers
# -----------------------
def gen_aps():
    xs=np.linspace(0,66,14); ys=np.linspace(0,44,8)
    coords=[(float(x),float(y)) for y in ys for x in xs]
    aps=[]; k=0
    for ch in CH24:
        tx=float(np.random.randint(14,21))
        aps.append(AP(f"AP24_{ch}","2.4",ch,20,tx,*coords[k],PHY_MAX_24)); k+=1
    for ch in CH5:
        w=20 if ch in DFS_CH and np.random.rand()<0.9 else int(np.random.choice(WIDTHS,p=[.7,.25,.05]))
        tx=float(np.random.randint(14,23))
        aps.append(AP(f"AP5_{ch}","5",ch,w,tx,*coords[k],PHY_MAX_5*width_scale(w))); k+=1
    return aps

def gen_clients(aps):
    cs=[]
    for a in aps:
        for i in range(CLIENTS_PER_AP):
            r=CELL_R*math.sqrt(np.random.rand()); t=np.random.rand()*2*math.pi
            cs.append(Client(f"C_{a.id}_{i}",a.id,a.x+r*math.cos(t),a.y+r*math.sin(t),r))
    return cs

# -----------------------
# RAM-optimized simulator (memmap + streaming CSVs)
# -----------------------
def run_simulation(hours=HOURS, scan=SCAN_PERIOD, sense=SENSE_TIME, mode=MODE, frac_real=FRAC_REAL):
    n_steps = int(hours * 3600 / scan)
    start = datetime.now()
    aps = gen_aps()
    clients = gen_clients(aps)
    num_aps = len(aps)
    total_samples = n_steps * num_aps
    log.info("Simulation: hours=%.2f scan=%.1f steps=%d aps=%d total_samples=%d",
             hours, scan, n_steps, num_aps, total_samples)

    # create memmap file for tensors (float32)
    memmap_path = OUT / "FFT_multiframe.dat"
    if memmap_path.exists(): memmap_path.unlink()
    # shape (N, T, F)
    mmap = np.memmap(memmap_path, dtype='float32', mode='w+', shape=(total_samples, FRAMES_PER_SAMPLE, FFT_BINS))
    log.info("Created memmap at %s shape=%s", memmap_path, mmap.shape)

    # prepare streaming CSV writers (write headers)
    fft_csv = OUT/"fft_dataset.csv"
    evt_csv = OUT/"sensing_events.csv"
    ap_csv = OUT/"Aplog_all_channels.csv"
    meta_csv = OUT/"FFT_multiframe_meta.csv"

    # header schemas
    fft_header = ["timestamp","ap_id","band","channel","label","center_mhz","bw_mhz","duty","power_dbm"] + [f"fft_bin_{i}" for i in range(LEGACY_BINS)]
    evt_header = ["timestamp","ap_id","band","channel","center_freq_mhz","bandwidth_mhz","duty_cycle","class_label","confidence"]
    ap_header = ["TIMESTAMP","AP_ID","BAND","CHANNEL","CHANNEL_WIDTH","TX_POWER_DBM","NOISE_FLOOR_DBM",
                 "AVG_CLIENT_SNR_DB","THROUGHPUT_AVG_Mbps","P95_RETRY_PCT","MEAN_QOE","UL_PER",
                 "BUSY_TIME","TOTAL_TIME","AIRTIME_UTILIZATION","NWIFI_DETECTED","NWIFI_TYPE","CLIENTS","MEAN_RSSI_DBM","CENTER_MHZ"]
    meta_header = ["idx","timestamp","ap_id","band","channel","label","center_mhz","bw_mhz","duty"]

    # write headers (overwrite if existing)
    pd.DataFrame(columns=fft_header).to_csv(fft_csv, index=False)
    pd.DataFrame(columns=evt_header).to_csv(evt_csv, index=False)
    pd.DataFrame(columns=ap_header).to_csv(ap_csv, index=False)
    pd.DataFrame(columns=meta_header).to_csv(meta_csv, index=False)

    sample_idx = 0
    # loop
    for step in range(n_steps):
        ts = (start + timedelta(seconds=step*scan)).isoformat()
        # random walk clients a bit
        for c in clients:
            c.x += np.random.normal(0,0.15); c.y += np.random.normal(0,0.15)
        for a in aps:
            nf = noise_floor_dbm(a.band, a.width)
            label = random.choices(["wifi","BLE","ZigBee","Microwave","FHSS","Radar"], weights=[0.5,0.1,0.1,0.1,0.1,0.1])[0]
            if a.band=="5" and a.ch in DFS_CH and np.random.rand() < 0.25:
                label = "Radar"
            center = ch2mhz(a.band, a.ch)
            ch_bw = np.random.uniform(1, min(40, a.width))
            ev_duty = np.random.uniform(0.03, 0.75)
            ev_power = nf + np.random.uniform(-40, 5)

            # decide real vs synthetic
            use_real = False
            if mode=="real":
                use_real=True
            elif mode=="hybrid":
                use_real = (np.random.rand() < frac_real) and (label in REAL_LIB)
            # generate tensor
            if use_real:
                tensor = generate_realistic_fft_from_real_data(label, center, ch_bw, nf)
            else:
                tensor = generate_highres_fft_synthetic(label, center, ch_bw, nf)

            # store in memmap
            mmap[sample_idx, :, :] = tensor  # writes directly to disk-backed memmap
            # legacy 64-bin downsample (mean across frames)
            bins = tensor.mean(axis=0)[:: FFT_BINS // LEGACY_BINS][:LEGACY_BINS]
            # append rows to CSVs (stream)
            fft_row = dict(timestamp=ts, ap_id=a.id, band=a.band, channel=a.ch, label=label,
                           center_mhz=center, bw_mhz=ch_bw, duty=ev_duty, power_dbm=ev_power)
            fft_row.update({f"fft_bin_{i}": float(bins[i]) for i in range(LEGACY_BINS)})
            pd.DataFrame([fft_row]).to_csv(fft_csv, mode='a', header=False, index=False)

            evt_row = dict(timestamp=ts, ap_id=a.id, band=a.band, channel=a.ch,
                           center_freq_mhz=center, bandwidth_mhz=ch_bw, duty_cycle=ev_duty,
                           class_label=label, confidence=float(np.clip(np.random.beta(8,2) if label!="wifi" else np.random.beta(2,8),0,1)))
            pd.DataFrame([evt_row]).to_csv(evt_csv, mode='a', header=False, index=False)

            # compute simple AP telemetry stats
            c_list = [c for c in clients if c.apid==a.id]
            if not c_list: c_list=[Client("temp",a.id,0,0,CELL_R*0.7)]
            snrs=[]; thrs=[]; rtys=[]; qoes=[]; pers=[]; rssis=[]
            for c in c_list:
                s_rssi = rssi_dbm(c.d,a) if 'rssi_dbm' in globals() else (noise_floor_dbm(a.band,a.width)-20)
                s_snr = s_rssi - nf
                r = np.clip(100/(1+math.exp(0.38*(s_snr-18))) + np.random.normal(0,2), 0, 100)
                th = max(0, ((PHY_MAX_24 if a.band=="2.4" else PHY_MAX_5)*width_scale(a.width) * np.clip((s_snr+5)/40,0,1)) * (0.9 + np.random.rand()*0.12))
                q = float(5*(0.7*np.clip((s_snr+5)/40,0,1) + 0.3*(1-np.clip(r/50,0,1))))
                p = float(100/(1+math.exp(0.3*(s_snr-15))))
                rssis.append(s_rssi); snrs.append(s_snr); thrs.append(th); rtys.append(r); qoes.append(q); pers.append(p)
            avg_snr = float(np.mean(snrs)); avg_thr = float(np.mean(thrs)); p95r = float(np.percentile(rtys,95)); mean_qoe = float(np.mean(qoes))
            avg_per = float(np.mean(pers)); mean_rssi = float(np.mean(rssis))
            busy = float(np.clip(scan*(0.25 + 0.75*0.5) + sense + scan*ev_duty*0.5, 0, scan))
            ap_row = dict(TIMESTAMP=ts, AP_ID=a.id, BAND=a.band, CHANNEL=a.ch, CHANNEL_WIDTH=a.width,
                          TX_POWER_DBM=a.tx, NOISE_FLOOR_DBM=nf, AVG_CLIENT_SNR_DB=avg_snr, THROUGHPUT_AVG_Mbps=avg_thr,
                          P95_RETRY_PCT=p95r, MEAN_QOE=mean_qoe, UL_PER=avg_per, BUSY_TIME=busy, TOTAL_TIME=scan,
                          AIRTIME_UTILIZATION=busy/scan*100, NWIFI_DETECTED=(label!="wifi"), NWIFI_TYPE=label,
                          CLIENTS=len(c_list), MEAN_RSSI_DBM=mean_rssi, CENTER_MHZ=ch2mhz(a.band,a.ch))
            pd.DataFrame([ap_row]).to_csv(ap_csv, mode='a', header=False, index=False)

            # meta
            meta_row = dict(idx=sample_idx, timestamp=ts, ap_id=a.id, band=a.band, channel=a.ch, label=label, center_mhz=center, bw_mhz=ch_bw, duty=ev_duty)
            pd.DataFrame([meta_row]).to_csv(meta_csv, mode='a', header=False, index=False)

            sample_idx += 1
            if sample_idx % 500 == 0:
                log.info("Generated %d / %d samples", sample_idx, total_samples)

    # flush memmap to disk
    mmap.flush()
    del mmap
    log.info("Simulation done. samples=%d memmap=%s", sample_idx, memmap_path)
    return str(memmap_path), total_samples

# -----------------------
# Training: Keras model using memmap via generator
# -----------------------
def make_dataset_from_memmap(memmap_path, meta_csv, total_samples, batch_size=BATCH_SIZE, shuffle=True):
    # open memmap readonly
    mmap = np.memmap(memmap_path, dtype='float32', mode='r', shape=(total_samples, FRAMES_PER_SAMPLE, FFT_BINS))
    meta = pd.read_csv(meta_csv)
    # labels -> integers
    labels, uniques = pd.factorize(meta['label'])
    class_names = list(uniques)
    # generator yields (x, y)
    def gen():
        for i in range(total_samples):
            x = mmap[i]  # shape (T, F)
            # normalize per-sample: z-score or min-max
            x = (x - x.mean()) / (x.std() + 1e-6)
            x = x.astype(np.float32)
            x = np.expand_dims(x, axis=-1)  # channel dim
            y = int(labels[i])
            yield x, y
    # build tf.data
    output_types = (tf.float32, tf.int32)
    output_shapes = ((FRAMES_PER_SAMPLE, FFT_BINS, 1), ())
    ds = tf.data.Dataset.from_generator(gen, output_types=output_types, output_shapes=output_shapes)
    if shuffle:
        ds = ds.shuffle(buffer_size=2048, seed=SEED)
    ds = ds.batch(batch_size).prefetch(tf.data.AUTOTUNE)
    return ds, class_names

def build_model(input_shape=(FRAMES_PER_SAMPLE, FFT_BINS, 1), num_classes=5):
    inputs = tf.keras.layers.Input(shape=input_shape)
    x = inputs
    # small conv stack
    x = tf.keras.layers.Conv2D(32, (3,7), padding='same', activation='relu')(x)
    x = tf.keras.layers.BatchNormalization()(x)
    x = tf.keras.layers.MaxPool2D((1,4))(x)  # reduce freq dim
    x = tf.keras.layers.Conv2D(64, (3,5), padding='same', activation='relu')(x)
    x = tf.keras.layers.BatchNormalization()(x)
    x = tf.keras.layers.MaxPool2D((2,4))(x)
    x = tf.keras.layers.Conv2D(128, (3,3), padding='same', activation='relu')(x)
    x = tf.keras.layers.GlobalAveragePooling2D()(x)
    x = tf.keras.layers.Dense(128, activation='relu')(x)
    outputs = tf.keras.layers.Dense(num_classes, activation='softmax')(x)
    model = tf.keras.Model(inputs, outputs)
    model.compile(optimizer=tf.keras.optimizers.Adam(LR),
                  loss='sparse_categorical_crossentropy',
                  metrics=['accuracy'])
    return model

# -----------------------
# Main entry (simulate then train)
# -----------------------
def main():
    parser = argparse.ArgumentParser()
    parser.add_argument("--hours", type=float, default=HOURS)
    parser.add_argument("--scan", type=float, default=SCAN_PERIOD)
    parser.add_argument("--sense", type=float, default=SENSE_TIME)
    parser.add_argument("--mode", type=str, choices=["synthetic","real","hybrid"], default=MODE)
    parser.add_argument("--frac_real", type=float, default=FRAC_REAL)
    args, unknown = parser.parse_known_args()

    memmap_path, total = run_simulation(hours=args.hours, scan=args.scan, sense=args.sense, mode=args.mode, frac_real=args.frac_real)
    meta_csv = OUT/"FFT_multiframe_meta.csv"

    # build dataset and split (80/20)
    meta = pd.read_csv(meta_csv)
    n = len(meta)
    log.info("Meta rows loaded: %d", n)
    # compute train/val indices (simple stratified split)
    from sklearn.model_selection import train_test_split
    train_idx, val_idx = train_test_split(meta.index.values, test_size=0.2, random_state=SEED, stratify=meta['label'])
    # create full dataset generator and then filter indices using TF dataset is cumbersome; simplest: build numpy index lists and create generator that yields only indices in set
    # We'll create two generators reading memmap directly
    labels, uniques = pd.factorize(meta['label'])
    class_names = list(uniques)
    num_classes = len(class_names)
    log.info("Classes: %s", class_names)

    mmap = np.memmap(memmap_path, dtype='float32', mode='r', shape=(total, FRAMES_PER_SAMPLE, FFT_BINS))

    def gen_from_indices(idxs):
        for i in idxs:
            x = mmap[i]
            x = (x - x.mean()) / (x.std() + 1e-6)
            x = np.expand_dims(x.astype(np.float32), axis=-1)
            y = int(labels[i])
            yield x, y

    # datasets
    train_ds = tf.data.Dataset.from_generator(lambda: gen_from_indices(train_idx),
                                              output_types=(tf.float32, tf.int32),
                                              output_shapes=((FRAMES_PER_SAMPLE, FFT_BINS, 1), ()))
    train_ds = train_ds.shuffle(2048, seed=SEED).batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)
    val_ds = tf.data.Dataset.from_generator(lambda: gen_from_indices(val_idx),
                                            output_types=(tf.float32, tf.int32),
                                            output_shapes=((FRAMES_PER_SAMPLE, FFT_BINS, 1), ()))
    val_ds = val_ds.batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)

    # build model
    model = build_model((FRAMES_PER_SAMPLE, FFT_BINS, 1), num_classes=num_classes)
    model.summary(print_fn=log.info)

    # callbacks
    checkpoint = tf.keras.callbacks.ModelCheckpoint(MODEL_DIR/"best_model.h5", save_best_only=True, monitor='val_loss')
    es = tf.keras.callbacks.EarlyStopping(monitor='val_loss', patience=6, restore_best_weights=True)
    reduce_lr = tf.keras.callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=3)

    # class weights to balance (optional)
    from sklearn.utils.class_weight import compute_class_weight
    class_weight = compute_class_weight('balanced', classes=np.unique(labels), y=labels)
    cw = {i: float(class_weight[i]) for i in range(len(class_weight))}
    log.info("Class weights: %s", cw)

    # train
    history = model.fit(train_ds, validation_data=val_ds, epochs=EPOCHS, callbacks=[checkpoint, es, reduce_lr], class_weight=cw)

    # evaluate & save final
    log.info("Training finished. Saving final model.")
    model.save(MODEL_DIR/"final_model.keras")
    # evaluation on val set
    val_preds = []
    val_trues = []
    for x_batch, y_batch in val_ds:
        preds = model.predict(x_batch)
        val_preds.extend(np.argmax(preds, axis=1).tolist())
        val_trues.extend(y_batch.numpy().tolist())
    from sklearn.metrics import classification_report
    report = classification_report(val_trues, val_preds, target_names=class_names, zero_division=0)
    log.info("Validation Classification Report:\n%s", report)
    # save report
    with open(MODEL_DIR/"val_report.txt", "w") as f:
        f.write(report)
    log.info("All done. Artifacts in %s", OUT)

if __name__=="__main__":
    main()


Epoch 1/30
   1026/Unknown 47s 41ms/step - accuracy: 0.6724 - loss: 1.0166

/usr/local/lib/python3.12/dist-packages/keras/src/trainers/epoch_iterator.py:160: UserWarning: Your input ran out of data; interrupting training. Make sure that your dataset or generator can generate at least `steps_per_epoch * epochs` batches. You may need to use the `.repeat()` function when building your dataset.
  self._interrupted_warning()


1026/1026 ━━━━━━━━━━━━━━━━━━━━ 57s 51ms/step - accuracy: 0.6724 - loss: 1.0165 - val_accuracy: 0.6803 - val_loss: 0.6207 - learning_rate: 0.0010
Epoch 2/30
1026/1026 ━━━━━━━━━━━━━━━━━━━━ 52s 50ms/step - accuracy: 0.6919 - loss: 0.9514 - val_accuracy: 0.6781 - val_loss: 0.6221 - learning_rate: 0.0010
Epoch 3/30
1025/1026 ━━━━━━━━━━━━━━━━━━━━ 0s 41ms/step - accuracy: 0.6850 - loss: 0.9561

1026/1026 ━━━━━━━━━━━━━━━━━━━━ 52s 50ms/step - accuracy: 0.6850 - loss: 0.9561 - val_accuracy: 0.6756 - val_loss: 0.6174 - learning_rate: 0.0010
Epoch 4/30
1026/1026 ━━━━━━━━━━━━━━━━━━━━ 52s 49ms/step - accuracy: 0.6831 - loss: 0.9452 - val_accuracy: 0.6795 - val_loss: 0.6181 - learning_rate: 0.0010
Epoch 5/30
1026/1026 ━━━━━━━━━━━━━━━━━━━━ 0s 41ms/step - accuracy: 0.6842 - loss: 0.9540

1026/1026 ━━━━━━━━━━━━━━━━━━━━ 106s 102ms/step - accuracy: 0.6842 - loss: 0.9540 - val_accuracy: 0.6952 - val_loss: 0.6118 - learning_rate: 0.0010
Epoch 6/30
1026/1026 ━━━━━━━━━━━━━━━━━━━━ 52s 49ms/step - accuracy: 0.6848 - loss: 0.9460 - val_accuracy: 0.7119 - val_loss: 0.6135 - learning_rate: 0.0010
Epoch 7/30
1025/1026 ━━━━━━━━━━━━━━━━━━━━ 0s 41ms/step - accuracy: 0.6843 - loss: 0.9447

1026/1026 ━━━━━━━━━━━━━━━━━━━━ 52s 50ms/step - accuracy: 0.6844 - loss: 0.9447 - val_accuracy: 0.7555 - val_loss: 0.6117 - learning_rate: 0.0010
Epoch 8/30
1026/1026 ━━━━━━━━━━━━━━━━━━━━ 52s 50ms/step - accuracy: 0.6932 - loss: 0.9465 - val_accuracy: 0.7534 - val_loss: 0.6131 - learning_rate: 0.0010
Epoch 9/30
1026/1026 ━━━━━━━━━━━━━━━━━━━━ 51s 49ms/step - accuracy: 0.6817 - loss: 0.9795 - val_accuracy: 0.7378 - val_loss: 0.6125 - learning_rate: 0.0010
Epoch 10/30
1026/1026 ━━━━━━━━━━━━━━━━━━━━ 51s 49ms/step - accuracy: 0.6960 - loss: 0.9489 - val_accuracy: 0.6800 - val_loss: 0.6170 - learning_rate: 0.0010
Epoch 11/30
1026/1026 ━━━━━━━━━━━━━━━━━━━━ 51s 49ms/step - accuracy: 0.6883 - loss: 0.9505 - val_accuracy: 0.7266 - val_loss: 0.6157 - learning_rate: 5.0000e-04
Epoch 12/30
1026/1026 ━━━━━━━━━━━━━━━━━━━━ 83s 49ms/step - accuracy: 0.6914 - loss: 0.9433 - val_accuracy: 0.7320 - val_loss: 0.6155 - learning_rate: 5.0000e-04
Epoch 13/30
1026/1026 ━━━━━━━━━━━━━━━━━━━━ 52s 50ms/step - accur

In [5]:
#!/usr/bin/env python3
import numpy as np
import pandas as pd
import tensorflow as tf
import psutil, time, json, os, math
from pathlib import Path
from sklearn.metrics import classification_report, confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns

# ----------------------------------------
# Config
# ----------------------------------------
DATA_DIR = Path("data")
MODEL_DIR = Path("model")
OUT_DIR = DATA_DIR / "non_wifi_classifier_results"
OUT_DIR.mkdir(exist_ok=True)

MEMMAP_PATH = DATA_DIR / "FFT_multiframe.dat"
META_PATH = DATA_DIR / "FFT_multiframe_meta.csv"
MODEL_PATH = MODEL_DIR / "best_model.h5"

FRAMES = 24
BINS = 512
BATCH_SIZE = 512  # tune this based on GPU memory

# ----------------------------------------
# GPU setup
# ----------------------------------------
gpus = tf.config.list_physical_devices("GPU")
if gpus:
    try:
        for g in gpus:
            tf.config.experimental.set_memory_growth(g, True)
        print(f"✅ Using GPU: {gpus}")
    except Exception as e:
        print("⚠️ Could not set GPU memory growth:", e)
device = "/GPU:0" if gpus else "/CPU:0"
print("Compute device:", device)

# ----------------------------------------
# Load data
# ----------------------------------------
print("Loading memmap + meta...")
meta = pd.read_csv(META_PATH)
total = len(meta)
print(f"Total samples: {total}")

mmap = np.memmap(
    MEMMAP_PATH,
    dtype="float32",
    mode="r",
    shape=(total, FRAMES, BINS)
)

labels, class_names = pd.factorize(meta["label"])
class_names = list(class_names)

# ----------------------------------------
# Build tf.data Dataset for batched inference
# ----------------------------------------
def sample_generator():
    """Yields (x, y) pairs in the original order."""
    for i in range(total):
        x = mmap[i]
        # per-sample normalization
        x = (x - x.mean()) / (x.std() + 1e-6)
        x = np.expand_dims(x.astype(np.float32), axis=-1)  # (T, F, 1)
        y = int(labels[i])
        yield x, y

output_types = (tf.float32, tf.int32)
output_shapes = ((FRAMES, BINS, 1), ())

ds = tf.data.Dataset.from_generator(
    sample_generator,
    output_types=output_types,
    output_shapes=output_shapes,
)

# parallel map/prefetch already inside generator, so just batch+prefetch
ds = ds.batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)

# ----------------------------------------
# Load model
# ----------------------------------------
print("Loading model...")
with tf.device(device):
    model = tf.keras.models.load_model(MODEL_PATH)

# ----------------------------------------
# Prediction + Metrics (batched + GPU)
# ----------------------------------------
print("Running batched inference...")

y_true = []
y_pred = []
y_conf = []

process = psutil.Process(os.getpid())
cpu_start = process.cpu_percent(interval=None)
mem_start = process.memory_info().rss
t0 = time.time()

num_steps = math.ceil(total / BATCH_SIZE)

with tf.device(device):
    for step, (x_batch, y_batch) in enumerate(ds, start=1):
        # model(x_batch, training=False) uses GPU/CPU in parallel over batch
        preds = model(x_batch, training=False).numpy()  # (batch, num_classes)

        y_true.extend(y_batch.numpy().tolist())
        y_pred.extend(np.argmax(preds, axis=1).tolist())
        y_conf.extend(np.max(preds, axis=1).tolist())

        print(f"[{step}/{num_steps}] processed batch of size {x_batch.shape[0]}")

t1 = time.time()
cpu_end = process.cpu_percent(interval=None)
mem_end = process.memory_info().rss

inference_time = (t1 - t0) / total * 1000.0  # ms per sample

# ----------------------------------------
# Classification report
# ----------------------------------------
report = classification_report(
    y_true,
    y_pred,
    target_names=class_names,
    zero_division=0
)
print("\n=== Classification Report ===")
print(report)

with open(OUT_DIR / "classification_report.txt", "w") as f:
    f.write(report)

# ----------------------------------------
# Confusion Matrix
# ----------------------------------------
cm = confusion_matrix(y_true, y_pred)
plt.figure(figsize=(8, 6))
sns.heatmap(
    cm,
    annot=True,
    fmt="d",
    cmap="Blues",
    xticklabels=class_names,
    yticklabels=class_names
)
plt.title("Confusion Matrix")
plt.xlabel("Predicted")
plt.ylabel("True")
plt.tight_layout()
plt.savefig(OUT_DIR / "confusion_matrix.png")
plt.close()

# ----------------------------------------
# CPU / RAM Budget
# ----------------------------------------
budget = {
    "cpu_percent_start": cpu_start,
    "cpu_percent_end": cpu_end,
    "ram_usage_mb": (mem_end - mem_start) / (1024 * 1024),
    "avg_inference_ms_per_sample": inference_time,
    "total_samples_eval": int(total),
    "batch_size": int(BATCH_SIZE),
    "device": device,
}

with open(OUT_DIR / "cpu_ram_budget.json", "w") as f:
    json.dump(budget, f, indent=4)

print("\n=== CPU/RAM Budget ===")
print(json.dumps(budget, indent=4))

# ----------------------------------------
# Export inference results with metadata
# ----------------------------------------
output = meta.copy()
output["true_label"] = [class_names[i] for i in y_true]
output["pred_label"] = [class_names[i] for i in y_pred]
output["confidence"] = y_conf

output[[
    "timestamp", "ap_id", "band", "channel",
    "true_label", "pred_label", "confidence",
    "duty", "center_mhz", "bw_mhz"
]].to_csv(OUT_DIR / "inference_export.csv", index=False)

print("\nSaved results to:", OUT_DIR)


⚠️ Could not set GPU memory growth: Physical devices cannot be modified after being initialized
Compute device: /GPU:0
Loading memmap + meta...
Total samples: 82080
Loading model...
Running batched inference...
[1/161] processed batch of size 512
[2/161] processed batch of size 512
[3/161] processed batch of size 512
[4/161] processed batch of size 512
[5/161] processed batch of size 512
[6/161] processed batch of size 512
[7/161] processed batch of size 512
[8/161] processed batch of size 512
[9/161] processed batch of size 512
[10/161] processed batch of size 512
[11/161] processed batch of size 512
[12/161] processed batch of size 512
[13/161] processed batch of size 512
[14/161] processed batch of size 512
[15/161] processed batch of size 512
[16/161] processed batch of size 512
[17/161] processed batch of size 512
[18/161] processed batch of size 512
[19/161] processed batch of size 512
[20/161] processed batch of size 512
[21/161] processed batch of size 512
[22/161] processed ba

In [6]:
#!/usr/bin/env python3
import numpy as np
import pandas as pd
import tensorflow as tf
import psutil, time, json, os, math, threading
from pathlib import Path
from sklearn.metrics import classification_report, confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns

# ---------------------------------------------------
# FORCE CPU ONLY
# ---------------------------------------------------
os.environ["CUDA_VISIBLE_DEVICES"] = ""  # disable GPU
print("🚫 GPU Disabled — running on CPU only.")

# ---------------------------------------------------
# Config
# ---------------------------------------------------
DATA_DIR = Path("data")
MODEL_DIR = Path("model")
OUT_DIR = DATA_DIR / "non_wifi_classifier_results_cpu"
OUT_DIR.mkdir(exist_ok=True)

MEMMAP_PATH = DATA_DIR / "FFT_multiframe.dat"
META_PATH = DATA_DIR / "FFT_multiframe_meta.csv"
MODEL_PATH = MODEL_DIR / "best_model.h5"

FRAMES = 24
BINS = 512
BATCH_SIZE = 256  # CPU-friendly size

# ---------------------------------------------------
# Load data
# ---------------------------------------------------
print("Loading memmap + meta...")
meta = pd.read_csv(META_PATH)
total = len(meta)
print("Total samples:", total)

mmap = np.memmap(
    MEMMAP_PATH,
    dtype="float32",
    mode="r",
    shape=(total, FRAMES, BINS)
)

labels, class_names = pd.factorize(meta["label"])
class_names = list(class_names)

# ---------------------------------------------------
# Build Dataset
# ---------------------------------------------------
def sample_gen():
    for i in range(total):
        x = mmap[i]
        x = (x - x.mean()) / (x.std() + 1e-6)
        x = np.expand_dims(x.astype(np.float32), axis=-1)
        y = int(labels[i])
        yield x, y

ds = tf.data.Dataset.from_generator(
    sample_gen,
    output_types=(tf.float32, tf.int32),
    output_shapes=((FRAMES, BINS, 1), ())
)

ds = ds.batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)

# ---------------------------------------------------
# Load model
# ---------------------------------------------------
print("Loading model...")
model = tf.keras.models.load_model(MODEL_PATH)

# ---------------------------------------------------
# Resource Monitoring Thread
# ---------------------------------------------------
process = psutil.Process(os.getpid())

cpu_samples = []
ram_samples = []
timing_samples = []

monitoring = True

def monitor_resources():
    while monitoring:
        cpu_samples.append(process.cpu_percent(interval=0.5))  # sample every 0.5 sec
        ram_samples.append(process.memory_info().rss / (1024*1024))  # MB

monitor_thread = threading.Thread(target=monitor_resources)
monitor_thread.start()

# ---------------------------------------------------
# Inference (batched)
# ---------------------------------------------------
y_true = []
y_pred = []
y_conf = []

print("Running CPU inference...")

num_batches = math.ceil(total / BATCH_SIZE)
batch_latencies = []

t0 = time.time()

for batch_idx, (x_batch, y_batch) in enumerate(ds, start=1):
    t_batch_start = time.time()

    pred_batch = model.predict(x_batch, verbose=0)

    t_batch_end = time.time()
    batch_latencies.append((t_batch_end - t_batch_start) / len(x_batch) * 1000.0)

    y_true.extend(y_batch.numpy().tolist())
    y_pred.extend(np.argmax(pred_batch, axis=1).tolist())
    y_conf.extend(np.max(pred_batch, axis=1).tolist())

    print(f"[Batch {batch_idx}/{num_batches}] {len(x_batch)} samples processed")

t1 = time.time()

monitoring = False
monitor_thread.join()

# ---------------------------------------------------
# Metrics
# ---------------------------------------------------
total_time_sec = (t1 - t0)
avg_ms = total_time_sec / total * 1000
lat_min = float(np.min(batch_latencies))
lat_max = float(np.max(batch_latencies))
lat_med = float(np.median(batch_latencies))

cpu_min = float(np.min(cpu_samples))
cpu_max = float(np.max(cpu_samples))
cpu_avg = float(np.mean(cpu_samples))

ram_min = float(np.min(ram_samples))
ram_max = float(np.max(ram_samples))
ram_avg = float(np.mean(ram_samples))
ram_delta = ram_max - ram_min

# ---------------------------------------------------
# Classification Output
# ---------------------------------------------------
report = classification_report(
    y_true, y_pred, target_names=class_names, zero_division=0
)
print("\n=== CLASSIFICATION REPORT ===")
print(report)

with open(OUT_DIR / "classification_report.txt", "w") as f:
    f.write(report)

# Confusion Matrix
cm = confusion_matrix(y_true, y_pred)
plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt="d", cmap="viridis",
            xticklabels=class_names,
            yticklabels=class_names)
plt.title("CPU Inference Confusion Matrix")
plt.tight_layout()
plt.savefig(OUT_DIR / "confusion_matrix.png")
plt.close()

# ---------------------------------------------------
# Save CPU/RAM Metrics
# ---------------------------------------------------
metrics = {
    "total_samples": total,
    "batch_size": BATCH_SIZE,

    "latency_ms_per_sample_avg": avg_ms,
    "latency_ms_min": lat_min,
    "latency_ms_max": lat_max,
    "latency_ms_median": lat_med,

    "cpu_percent_min": cpu_min,
    "cpu_percent_max": cpu_max,
    "cpu_percent_avg": cpu_avg,

    "ram_usage_min_mb": ram_min,
    "ram_usage_max_mb": ram_max,
    "ram_usage_avg_mb": ram_avg,
    "ram_delta_mb": ram_delta,

    "total_inference_time_sec": total_time_sec,
}

with open(OUT_DIR / "cpu_ram_detailed.json", "w") as f:
    json.dump(metrics, f, indent=4)

# ---------------------------------------------------
# Export inference CSV
# ---------------------------------------------------
output = meta.copy()
output["true_label"] = [class_names[i] for i in y_true]
output["pred_label"] = [class_names[i] for i in y_pred]
output["confidence"] = y_conf

output[[
    "timestamp","ap_id","band","channel",
    "true_label","pred_label","confidence",
    "duty","center_mhz","bw_mhz"
]].to_csv(OUT_DIR / "inference_export.csv", index=False)

print("\n✔ CPU inference complete. Results saved to:", OUT_DIR)


🚫 GPU Disabled — running on CPU only.
Loading memmap + meta...
Total samples: 82080
Loading model...


Running CPU inference...
[Batch 1/321] 256 samples processed
[Batch 2/321] 256 samples processed
[Batch 3/321] 256 samples processed
[Batch 4/321] 256 samples processed
[Batch 5/321] 256 samples processed
[Batch 6/321] 256 samples processed
[Batch 7/321] 256 samples processed
[Batch 8/321] 256 samples processed
[Batch 9/321] 256 samples processed
[Batch 10/321] 256 samples processed
[Batch 11/321] 256 samples processed
[Batch 12/321] 256 samples processed
[Batch 13/321] 256 samples processed
[Batch 14/321] 256 samples processed
[Batch 15/321] 256 samples processed
[Batch 16/321] 256 samples processed
[Batch 17/321] 256 samples processed
[Batch 18/321] 256 samples processed
[Batch 19/321] 256 samples processed
[Batch 20/321] 256 samples processed
[Batch 21/321] 256 samples processed
[Batch 22/321] 256 samples processed
[Batch 23/321] 256 samples processed
[Batch 24/321] 256 samples processed
[Batch 25/321] 256 samples processed
[Batch 26/321] 256 samples processed
[Batch 27/321] 256 sam